# Pixelle-Video · Scene by Scene × Wan2GP on Google Colab

Runs the **step-by-step (scene-by-scene)** variant of Pixelle-Video: instead of one fully-automatic `topic → final video` call, **you review and control the input/output of every stage and every scene**:

```
① Script    AI writes the narrations  →  you edit / ✨AI-rewrite / add / delete
② Prompts   AI writes per-scene media prompts  →  you edit / 🎲regenerate
③ Scenes    per scene: 🎤 audio → 🖼️/🎬 image-or-video → 🎞️ segment
            (preview each output inline, regenerate any piece)
④ Final     compose all segments (+ optional BGM) → final.mp4
```

Media is generated with the **Wan2GP in-process backend** (models loaded directly in this runtime through WanGP's Python API). No ComfyUI server, no RunningHub key.

Run the cells in order. Dependencies are installed **once** and shared by everything: the step-by-step cells, the Streamlit wizard UI, and plain `wgp.py`.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. The defaults below are sized for it: **Z-Image Turbo 6B** for images and **Wan 2.1 1.3B** for video clips. On a bigger GPU (L4/A100) switch to `wan2gp/image_qwen.json`, `wan2gp/video_wan2.1_fusionx.json` or `wan2gp/video_ltx2_distilled.json`.

> **LLM note:** Pixelle-Video needs an OpenAI-compatible LLM endpoint (Qwen/DashScope, DeepSeek, OpenAI, Ollama, your own vLLM tunnel, ...) to write the script and media prompts. Fill it in at step 6.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime -> Change runtime type, select GPU, save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path

Choose where Wan2GP should be installed. It bundles both Pixelle-Video flavors:
- `Pixelle_video/` — the original fully-automatic app (also holds the shared config / workflows / templates / output)
- `Pixelle_video_scene_by_scene/` — the step-by-step engine + wizard UI used by this notebook


In [ ]:
from pathlib import Path

WAN2GP_ROOT  = Path('/content/wan2gp').resolve()
PIXELLE_ROOT = WAN2GP_ROOT / 'Pixelle_video'
SBS_ROOT     = WAN2GP_ROOT / 'Pixelle_video_scene_by_scene'
print(f'Wan2GP will be installed to:   {WAN2GP_ROOT}')
print(f'Pixelle-Video (core + data):   {PIXELLE_ROOT}')
print(f'Scene-by-Scene app:            {SBS_ROOT}')


## 3. Download or update Wan2GP

Clone the repository on the `feature/pixelle-step-by-step` branch (the one that contains `Pixelle_video_scene_by_scene/`); pull the latest changes if it already exists.


In [ ]:
import subprocess

repo_url = 'https://github.com/hoangthvn2201/Wan2GP'
branch = 'feature/pixelle-step-by-step'

if WAN2GP_ROOT.exists():
    print('Repository already exists. Updating...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', branch], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', 'origin', branch], check=True)


## 4. Install system dependencies

Shared libraries for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.


In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


## 5. Install Python dependencies (one-time)

A **single install** covers everything:

- `requirements.txt` (Wan2GP) — torch ecosystem, diffusers, loguru, pydantic, moviepy, ffmpeg-python, ...
- `Pixelle_video/requirements.txt` — only the Pixelle extras (streamlit, openai, edge-tts, comfykit, playwright, ...)

The scene-by-scene app has **no extra dependencies** of its own — it reuses the Pixelle-Video core. Afterwards Chromium is installed for the HTML frame-template rendering. This cell takes several minutes.


In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers==0.0.32.post2',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

# One-time install: Wan2GP requirements + Pixelle-Video extras in a single resolve
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    '-r', str(PIXELLE_ROOT / 'requirements.txt')], check=True, env=env)

# Chromium for HTML frame template rendering (Pixelle composes subtitles via Playwright)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True, env=env)

# Re-assert a modern setuptools AFTER all installs: the dependency resolve can
# remove it from /usr/local, letting Ubuntu's ancient system pkg_resources
# (which still uses pkgutil.ImpImporter, removed in Python 3.12) shadow it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend.


In [ ]:
from pathlib import Path

target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 6. Configure Pixelle-Video

Fill in your **LLM endpoint** (required) and adjust the media workflows / TTS voice if you like, then run the cell — it writes `Pixelle_video/config.yaml` (shared by both apps).

The `wan2gp/...` workflows are *descriptors* that map to WanGP models (see `Pixelle_video/WAN2GP_BACKEND.md`). Model checkpoints are downloaded automatically by WanGP on first use.


In [ ]:
import yaml

# --- LLM (required: writes the narration script & media prompts) ------------
LLM_API_KEY  = ''                                  # <-- your API key
LLM_BASE_URL = 'https://api.deepseek.com'          # any OpenAI-compatible endpoint
LLM_MODEL    = 'deepseek-chat'

# --- Media generation (wan2gp = models loaded in-process by WanGP) ---------
IMAGE_WORKFLOW = 'wan2gp/image_z_image.json'       # Z-Image Turbo 6B  (T4-friendly)
VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_1.3B.json'   # Wan 2.1 1.3B t2v  (T4-friendly)
# Bigger GPUs:
#   IMAGE_WORKFLOW = 'wan2gp/image_qwen.json'             # Qwen Image 20B
#   VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_fusionx.json'   # Wan 2.1 FusioniX 14B
#   VIDEO_WORKFLOW = 'wan2gp/video_ltx2_distilled.json'   # LTX-2 22B (video + audio)

PROMPT_PREFIX = ('Minimalist black-and-white matchstick figure style illustration, '
                 'clean lines, simple sketch style')

# --- TTS (local edge-tts, runs on CPU) --------------------------------------
TTS_VOICE = 'en-US-GuyNeural'    # e.g. 'zh-CN-YunjianNeural' for Chinese
TTS_SPEED = 1.2

# --- WanGP runtime -----------------------------------------------------------
WAN2GP_CLI_ARGS = ['--profile', '5']   # low-VRAM profile (same as the official notebook)

config = {
    'project_name': 'Pixelle-Video',
    'llm': {
        'api_key': LLM_API_KEY,
        'base_url': LLM_BASE_URL,
        'model': LLM_MODEL,
        'enable_thinking': False,
    },
    'comfyui': {
        'comfyui_url': 'http://127.0.0.1:8188',
        'comfyui_api_key': None,
        'runninghub_api_key': None,
        'runninghub_concurrent_limit': 1,
        'runninghub_instance_type': None,
        'tts': {
            'inference_mode': 'local',
            'local': {'voice': TTS_VOICE, 'speed': TTS_SPEED},
            'comfyui': {'default_workflow': None},
        },
        'image': {'default_workflow': IMAGE_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
        'video': {'default_workflow': VIDEO_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
    },
    'wan2gp': {
        'root': str(WAN2GP_ROOT),
        'cli_args': WAN2GP_CLI_ARGS,
        'output_dir': None,
    },
    'template': {'default_template': '1080x1920/image_default.html'},
}

(PIXELLE_ROOT / 'config.yaml').write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print('config.yaml written:')
print((PIXELLE_ROOT / 'config.yaml').read_text())


## 7. Initialize the core + the Scene-by-Scene engine

Sets up paths, initializes the core services and creates the `SceneBySceneEngine`. The WanGP session itself is created lazily — model weights are only loaded (and downloaded) on the first generation.


In [ ]:
import os, sys

os.chdir(PIXELLE_ROOT)                                   # relative paths: workflows/, templates/, output/
os.environ['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)

# sys.path: scene-by-scene app first (sbs), then core + Wan2GP root
for p in (str(WAN2GP_ROOT), str(PIXELLE_ROOT), str(SBS_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

from pixelle_video.service import pixelle_video
from sbs import SceneBySceneEngine

await pixelle_video.initialize()
engine = SceneBySceneEngine(pixelle_video)
print(pixelle_video)
print('wan2gp media workflows:', [w for w in pixelle_video.media.available if w.startswith('wan2gp/')])


## 8. Step ① — Generate the script (then review it)

Enter a topic and run. The LLM writes one narration per scene plus a title — **nothing else is generated yet**.

> Template naming controls the media type: `image_*.html` templates use the image model, `video_*.html` templates use the video model, `static_*.html` use no media model at all.


In [ ]:
TOPIC    = 'Why should you build a daily reading habit?'
N_SCENES = 3
TEMPLATE = '1080x1920/image_default.html'   # try '1080x1920/video_default.html' for AI video scenes

from pixelle_video.utils.template_util import resolve_template_path, get_template_type
from pixelle_video.services.frame_html import HTMLFrameGenerator

template_type = get_template_type(TEMPLATE.split('/')[-1])    # 'static' | 'image' | 'video'
media_workflow = {'image': IMAGE_WORKFLOW, 'video': VIDEO_WORKFLOW}.get(template_type)
media_width, media_height = HTMLFrameGenerator(resolve_template_path(TEMPLATE)).get_media_size()
print(f'Template type: {template_type} | media workflow: {media_workflow} | media size: {media_width}x{media_height}')
print()

title, narrations = await engine.generate_script(
    text=TOPIC,
    mode='generate',          # or 'fixed' to split a ready-made script (use split_mode=...)
    n_scenes=N_SCENES,
)

print(f'Title: {title}')
for i, n in enumerate(narrations, 1):
    print(f'  Scene {i}: {n}')


### Edit the script (optional)

This is the whole point of the step-by-step flow — adjust anything before moving on:

- **manual edit**: assign directly into the `narrations` list (or change `title`)
- **✨ AI rewrite**: ask the LLM to rewrite one scene, optionally with an instruction
- **add / remove scenes**: regular Python list operations


In [ ]:
# --- Manual edits (uncomment & adapt) ---------------------------------------
# title = 'My better title'
# narrations[0] = 'My own opening line for scene 1.'
# narrations.append('One extra closing scene.')
# del narrations[2]

# --- AI rewrite of a single scene (uncomment to use) -------------------------
# narrations[0] = await engine.rewrite_narration(
#     narrations[0], topic=TOPIC,
#     instruction='make it a question that hooks the viewer',
# )

print(f'Title: {title}')
for i, n in enumerate(narrations, 1):
    print(f'  Scene {i}: {n}')


## 9. Step ② — Generate the media prompts (then review them)

One media-generation prompt per scene, with the style `PROMPT_PREFIX` already applied. Skipped automatically for `static_*` templates.


In [ ]:
if template_type == 'static':
    prompts = [None] * len(narrations)
    print('Static template — no media prompts needed.')
else:
    prompts = await engine.generate_prompts(narrations, prompt_prefix=PROMPT_PREFIX)
    for i, p in enumerate(prompts, 1):
        print(f'Scene {i}: {p}\n')


In [ ]:
# --- Edit prompts before generation (uncomment & adapt) ----------------------
# prompts[0] = 'a cozy reading nook by a rainy window, warm lamp light, ' + PROMPT_PREFIX

# --- Or let the AI regenerate a single one ------------------------------------
# prompts[1] = await engine.generate_prompt_for(narrations[1], prompt_prefix=PROMPT_PREFIX)

for i, p in enumerate(prompts, 1):
    print(f'Scene {i}: {p}\n')


## 10. Create the project

Freezes the script + prompts + settings into a `SceneProject` with its own task directory (`Pixelle_video/output/<task_id>/`). Each scene gets a stable `uid` so its assets can be regenerated safely.


In [ ]:
params = {
    'text': TOPIC,
    'mode': 'generate',
    'n_scenes': len(narrations),
    'split_mode': 'paragraph',
    'title': title,
    'tts_inference_mode': 'local',
    'tts_voice': TTS_VOICE,
    'tts_speed': TTS_SPEED,
    'frame_template': TEMPLATE,
    'template_params': None,
    'media_workflow': media_workflow,
    'prompt_prefix': PROMPT_PREFIX,
    'media_width': media_width,
    'media_height': media_height,
}

project = engine.create_project(title, narrations, prompts, params)
print(f'Task: {project.task_id}  ({len(project.scenes)} scenes, media={project.media_requirement})')
print(f'Dir:  {project.task_dir}')


## 11. Step ③ — Generate scene 1 piece by piece

Each sub-step produces a previewable output. Re-run any cell to regenerate just that piece.

> The **first** media generation downloads the model checkpoint (a few GB) — subsequent scenes reuse the model already loaded in VRAM.


In [ ]:
# 🎤 Audio (TTS) — the narration voiceover; its duration drives video-clip length
from IPython.display import Audio, display

scene = project.scenes[0]
await engine.generate_audio(project, scene, 0)
print(f'Duration: {scene.duration:.2f}s')
display(Audio(scene.audio_path))


In [ ]:
# 🖼️/🎬 Media (WanGP, in-process) — skipped for static templates
from IPython.display import Image, Video, display

if project.needs_media:
    await engine.generate_media(project, scene, 0)
    if scene.media_type == 'video':
        display(Video(scene.video_path, embed=True, width=300))
    else:
        display(Image(scene.image_path, width=300))
else:
    print('Static template — no media for this scene.')


In [ ]:
# 🎞️ Segment — subtitled HTML frame + audio, the final building block of the video
from IPython.display import Video, display

await engine.render_segment(project, scene, 0)
display(Video(scene.segment_path, embed=True, width=300))


### Not happy with scene 1? Regenerate any piece

Edit the text and re-run only what changed — invalidation is automatic:


In [ ]:
# --- Change the narration (invalidates audio + segment) ----------------------
# scene.narration = 'A brand new opening line.'
# scene.invalidate_audio()
# await engine.generate_audio(project, scene, 0)

# --- Change the prompt (invalidates media + segment) --------------------------
# scene.prompt = 'a towering library at golden hour, ' + PROMPT_PREFIX
# scene.invalidate_media()
# await engine.generate_media(project, scene, 0)

# --- Then re-render the segment -----------------------------------------------
# await engine.render_segment(project, scene, 0)

print('Scene 1 status:',
      'audio ✅' if scene.audio_path else 'audio ⬜',
      '| media ✅' if scene.has_media else '| media ⬜',
      '| segment ✅' if scene.segment_path else '| segment ⬜')


## 12. Generate the remaining scenes

`process_scene` runs whatever is still missing for each scene (audio → media → segment). Re-run this cell safely — finished scenes are skipped.


In [ ]:
for i, sc in enumerate(project.scenes):
    if sc.segment_path:
        print(f'Scene {i+1}: already done ✅')
        continue
    await engine.process_scene(
        project, sc, i,
        progress_callback=lambda stage, i=i: print(f'  Scene {i+1}: {stage}...'),
    )
    print(f'Scene {i+1}: done ✅ ({sc.duration:.1f}s)')

print()
print(f'Segments ready: {sum(1 for s in project.scenes if s.segment_path)}/{len(project.scenes)}')


### Review all segments (optional)


In [ ]:
from IPython.display import Video, display

for i, sc in enumerate(project.scenes, 1):
    print(f'Scene {i}: {sc.narration}')
    display(Video(sc.segment_path, embed=True, width=260))


## 13. Step ④ — Compose the final video

Concatenates all segments and (optionally) adds background music, then persists the task so it appears in the web UI's History page.


In [ ]:
BGM = None            # or e.g. 'default.mp3' (any file in Pixelle_video/bgm/)
BGM_VOLUME = 0.2

result = await engine.compose_final(project, bgm_path=BGM, bgm_volume=BGM_VOLUME)

print(f"Final video: {result['video_path']}")
print(f"Duration:    {result['duration']:.1f}s | Size: {result['file_size'] / 1e6:.1f} MB | Scenes: {result['n_scenes']}")


### Preview the result


In [ ]:
from IPython.display import Video

Video(result['video_path'], embed=True, width=320)


## 14. (Optional) Launch the Scene-by-Scene Web UI

The same step-by-step flow as a 5-step **Streamlit wizard** (Setup → Script → Prompts → Scenes → Final), exposed through a free Cloudflare quick tunnel. Click the printed `trycloudflare.com` link; keep the cell running while you use the UI and press **Stop** when done.

It shares the same config / output as this notebook, so videos composed in the UI also land in `Pixelle_video/output/`.


In [ ]:
import os, re, subprocess, sys

# Cloudflare quick tunnel binary
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

env = os.environ.copy()
env['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)
env['PYTHONPATH'] = f"{SBS_ROOT}:{WAN2GP_ROOT}:{PIXELLE_ROOT}"

streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(SBS_ROOT / 'web' / 'app.py'),
     '--server.port', '8502', '--server.headless', 'true'],
    cwd=str(PIXELLE_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8502', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print('Waiting for the tunnel URL...')
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n🌐 Scene-by-Scene Web UI: {match.group(0)}\n')
        break

try:
    for line in iter(streamlit_proc.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping...')
finally:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print('Web UI and tunnel stopped.')


## Notes & troubleshooting

- **Outputs** land in `Pixelle_video/output/<task_id>/final.mp4`; per-scene assets in `frames/<scene_uid>_*.{mp3,png,mp4}` (uid-based, so regenerating never collides).
- **Regeneration rules**: changing a narration invalidates that scene's audio + segment; changing a prompt invalidates its media + segment. `process_scene` only re-runs what's missing.
- **Video templates**: the clip length is synced to the narration audio, so always generate the **audio before the media** for `video_*` templates (the cells above already do).
- **First generation is slow**: WanGP downloads the model checkpoint, then keeps it loaded in VRAM — later scenes are much faster.
- **Switching models** (e.g. an `image_*` run followed by a `video_*` run) makes WanGP unload/reload models — expected.
- **Out of VRAM / RAM on T4**: stick to `image_z_image` + `video_wan2.1_1.3B`, keep `--profile 5`, and use smaller media sizes (the template's media size is capped automatically by each descriptor's `max_pixels`).
- **Reasoning LLMs** (MiniMax-M3, DeepSeek-R1, Qwen3, ...): hidden chain-of-thought counts against the token budget, which can yield `LLM returned no content` on short completions. Pixelle retries once with a larger budget automatically.
- **`AttributeError: module 'pkgutil' has no attribute 'ImpImporter'`** when launching the UI: an old system `pkg_resources` is shadowing the modern one on Python 3.12. Run `pip install --upgrade setuptools wheel` and restart (the install cell now does this automatically).
- Full backend documentation: `Pixelle_video/WAN2GP_BACKEND.md` · app documentation: `Pixelle_video_scene_by_scene/README.md`.
